## XBRL US API - FERC schedule by report  

### Authenticate for access token 
Click in the gray code cell below, then click the Run button above to execute the cell. Type your XBRL US Web account email, account password, Client ID, and secret as noted, pressing the Enter key on the keyboard after each entry.

XBRL US limits records returned for a query to improve efficiency; this script loops to collect all data from the Public Filings Database for a query. **Non-members might not be able to return all data for a query** - join XBRL US for comprehensive access - https://xbrl.us/join.

In [ ]:
print('Enter your XBRL US Web account email: ')
import os, re, sys, json
import requests
import pandas as pd
from IPython.display import display, HTML
import numpy as np
import getpass
from datetime import datetime
import urllib
from urllib.parse import urlencode
email = input()
password = getpass.getpass(prompt='Password: ')
clientid = getpass.getpass(prompt='Client ID: ')
secret = getpass.getpass(prompt='Secret: ')

body_auth = {'username' : ''.join(email), 
            'client_id': ''.join(clientid), 
            'client_secret' : ''.join(secret), 
            'password' : ''.join(password), 
            'grant_type' : 'password', 
            'platform' : 'ipynb' }

payload = urlencode(body_auth)
url = 'https://api.xbrl.us/oauth2/token'
headers = {"Content-Type": "application/x-www-form-urlencoded"}

res = requests.request("POST", url, data=payload, headers=headers)
auth_json = res.json()

if 'error' in auth_json:
    print ("\n\nThere was a problem generating an access token with these credentials. Run the first cell again to enter credentials.")
else:
    print ("\n\nYour access token expires in 60 minutes. After it expires, run the cell immediately below this one to generate a new token and continue to use the query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
access_token = auth_json['access_token']
refresh_token = auth_json['refresh_token']
newaccess = ''
newrefresh = ''
#print('access token: ' + access_token + ' refresh token: ' + refresh_token)

#### Refresh token 
The cell below is only needed to refresh an expired access token after 60 minutes. When the access token no longer returns results, run the cell below to refresh the access token or re-enter credentials by running the cell above. Until the refresh token process is needed, **skip ahead to _Make a Query_**. 


In [ ]:
token = token if newrefresh != '' else refresh_token 

refresh_auth = {'client_id': ''.join(clientid), 
            'client_secret' : ''.join(secret), 
            'grant_type' : 'refresh_token', 
            'platform' : 'ipynb', 
            'refresh_token' : ''.join(token) }
refreshres = requests.post(url, data=refresh_auth)
refresh_json = refreshres.json()
access_token = refresh_json['access_token']
refresh_token = refresh_json['refresh_token']#print('access token: ' + access_token + 'refresh token: ' + refresh_token)
print('Your access token is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.')
print(access_token)

### Make a query 
After the access token confirmation appears above, you can modify the query below and use the **_Cell >> Run_** menu option with the cell **immediately below this text** to run the query for updated results. 

The sample results are from 10+ years of data for companies filing data on the _Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion_ , and may take several minutes to recreate.  To test for results quickly, modify the **_report\_ids_** to shorten the list, and change the **_XBRL\_Elements_** to return different data from a FERC Form 1 Schedule.
  
Refer to XBRL API documentation at https://xbrlus.github.io/xbrl-api/#/Facts/getFactDetails for other endpoints and parameters to filter and return. 

In [ ]:
# Define the parameters for the filter and fields to be returned, 
# run the loop to return results
offset_value = 0
res_df = []

# Define the parameters of the query

XBRL_Elements = ["200 - Schedule - Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion"]

# query for unique Form 1 reports, sort by year descending and name ascending
# https://api.xbrl.us/api/v1/fact/search?unique&report.source-name=ferc&report.document-type=1&fact.has-dimensions=false&concept.local-name=CompanyIdentifier&fields=period.fiscal-year.sort(DESC),entity.name.sort(ASC),fact.value,report.id

report_ids = [
'568937', '570499', '570510', '570501', '570507', '570504', '568947', '563267', '563272', '573288', 
'564870', '571058', '570426', '564792', '563256', '563278', '563213', '569115', '568436', '570532', 
'563364', '564700', '569108', '563355', '563352', '563358', '563346', '568435', '563114', '568431', 
'561458', '563112', '569845', '569738', '564872', '570272', '570679', '563307', '563361', '568432', 
'564698', '563340', '569686', '570242', '568219', '563304', '568551', '570448', '570263', '563124', 
'569286', '570544', '570405', '570408', '569871', '563216', '569301', '570388', '570490', '563230', 
'563207', '570484', '569827', '567685', '567681', '563241', '570458', '569324', '563328', '568430', 
'568875', '564788', '570282', '570542', '569274', '570396', '570461', '563334', '568020', '563128', 
'563316', '563310', '569282', '569212', '569214', '569284', '563963', '563367', '569292', '567753',
'569198', '570325', '568426', '569136', '570862', '569326', '570016', '570516', '571336', '570390',
'570033', '569276', '569863', '563349', '567455', '570022', '570479', '567679', '569737', '570482', 
'570305', '568425', '568403', '563331', '563325', '569204', '570540', '570284', '570467', '563319', 
'563322', '570525', '563337', '563236', '568221', '568224', '568220', '568222', '568223', '569809', 
'563284', '569630', '570010', '568611', '563264', '563275', '563287', '570008', '563136', '570677', 
'569208', '569306', '569493', '569304', '564378', '566443', '563248', '570181', '570183', '563313', 
'568429', '570028', '569807', '564694', '570020', '569140', '570285', '461008', '477084', '461384', 
'477047', '461149', '477074', '461151', '477076', '461153', '477078', '461146', '477069', '461148', 
'477072', '461378', '477042', '461005', '477080', '462306', '462443', '496543', '496544', '462405', 
'460125', '544461', '460121', '544462', '463398', '459074', '461382', '477046', '459002', '460493', 
'461460', '459035', '461159', '469148', '461450', '461466', '461495', '463129', '462512', '461492',
'459066', '461451', '465113', '461389', '462510', '460104', '459042', '459043', '474317', '462494', 
'462203', '459019', '484164', '462353', '467188', '459044', '461141', '461475', '462200', '459752', 
'462439', '462365', '463131', '465913', '462496', '462470', '479183', '462468', '479184', '462456', 
'465809', '461023', '459020', '568614', '461404', '461403', '462262', '488429', '461402', '488427', 
'461401', '462259', '488041', '461400', '462361', '462355', '563205', '462351', '563203', '462345', 
'462336', '461398', '459058', '461454', '460992', '461363', '474322', '462179', '462340', '462348', 
'461154', '462417', '459005', '461409', '544459', '476709', '461383', '477048', '461457', '459024', 
'460117', '461386', '459026', '459027', '459028', '459068', '460103', '477057', '459030', '461381', 
'477045', '481622', '476705', '459010', '459029', '461433', '462181', '459070', '474318', '474319', 
'459050', '462338', '459072', '459025', '462199', '459356', '459075', '462458', '466273', '460492',
'462264', '459078', '461144', '476704', '459009', '466272', '461422', '462370', '462386', '462381', 
'459051', '459061', '461410', '462308', '463397', '462393', '462411', '461439', '482573', '462282', 
'461425', '461426', '458996', '466485', '461453', '459064', '461415', '477044', '476707', '461156', 
'459063', '462492', '462476', '461484', '460757', '461155', '458331', '461356', '477086', '459017', 
'459077', '459071', '462330', '544431', '460469', '460470', '460890', '463529', '459045', '459041', 
'461463', '461424', '461408', '462383', '461413', '477089', '481432', '461420', '461482', '461434', 
'462304', '462480', '462274', '461445', '461438', '482152', '462198', '459885', '496850', '462328', 
'462770', '462389', '544445', '461477', '479228', '462516', '462180', '462400', '461411', '477082', 
'481430', '461399', '461377', '459073', '459065', '461405', '461407', '461376', '462478', '461412', 
'459069', '459067', '462302', '459076', '461368', '461352', '477059', '461358', '477067', '461355',
'477061', '461353', '477063', '461472', '459015', '461277', '461359', '459059', '459016', '461470', 
'460487', '466276', '461390', '461393', '461392', '477065', '462462', '461373', '496851', '460980', 
'460978', '459079', '461380', '477043', '462498', '461469', '460116', '461464', '461435', '462398', 
'465114', '477053', '544400', '474320', '428080', '428084', '428088', '428092', '428096', '428100', 
'428104', '482577', '428111', '482624', '428115', '428119', '482625', '482628', '428127', '482629', 
'482630', '482631', '482632', '482633', '482634', '482635', '428150', '428154', '482636', '482637', 
'428164', '428543', '482638', '482639', '428174', '428550', '428181', '428185', '482640', '428192', 
'482641', '428199', '482644', '428567', '482645', '428210', '482646', '428217', '428221', '482647', 
'428228', '428230', '482648', '482651', '482655', '482659', '482663', '428238', '482667', '482671', 
'482672', '482673', '482674', '482675', '482676', '482677', '482678', '428264', '428617', '487501', 
'563222', '487502', '563219', '428620', '428623', '428626', '428280', '428284', '482679', '482682', 
'428292', '428295', '482683', '428302', '428306', '428310', '482684', '428648', '482688', '428653', 
'482689', '482690', '428329', '482691', '482692', '428339', '428343', '428347', '482693', '482694', 
'482695', '482696', '428680', '428366', '428370', '482697', '482698', '428380', '428693', '428387', 
'428699', '428391', '428395', '428396', '482699', '428403', '428407', '482702', '482703', '482704', 
'428418', '428425', '482705', '428432', '428436', '428440', '482706', '428447', '428451', '482707', 
'428458', '428740', '482708', '428465', '482709', '482710', '482711', '482712', '428481', '482715', 
'482716', '482717', '428763', '482718', '482719', '482720', '482721', '482725', '482726', '482727', 
'428790', '544430', '482728', '482729', '482730', '482731', '482734', '482737', '482738', '482739',
'482740', '482741', '482742', '482743', '482747', '482748', '482749', '482750', '482751', '482752', 
'482753', '482754', '482755', '482756', '482757', '482758', '482759', '482762', '482763', '482764', 
'496783', '482765', '482766', '428913', '482767', '482768', '482769', '428926', '427260', '427264', 
'427268', '427272', '427276', '427280', '427284', '427901', '427291', '427905', '427295', '427298', 
'427906', '427910', '427306', '427911', '427912', '427913', '427914', '427915', '427916', '427917', 
'427328', '427918', '427919', '427338', '427920', '427922', '427923', '427347', '427351', '427355', 
'427356', '427924', '427363', '427925', '427370', '427929', '427374', '427930', '427381', '427931', 
'427388', '427392', '427932', '427399', '427933', '427936', '427940', '427944', '427947', '427948', 
'427952', '427956', '427957', '427958', '427959', '427960', '427961', '427962', '427963', '427964', 
'427433', '427965', '544232', '544229', '427966', '427967', '427968', '427441', '427445', '427972', 
'427976', '427449', '427453', '427977', '427460', '427464', '427978', '427980', '427981', '427982', 
'427983', '427482', '427986', '427987', '427490', '427494', '427498', '427988', '427989', '427990', 
'427991', '427520', '427524', '427528', '427992', '427993', '427538', '427542', '427994', '428698', 
'427995', '427552', '427553', '427996', '427560', '427564', '428000', '428001', '428002', '427574', 
'427581', '428003', '427588', '427592', '427596', '428004', '427603', '428005', '428006', '427613', 
'428007', '427620', '428008', '428010', '428011', '428012', '427635', '428016', '428017', '428018', 
'428019', '428020', '427651', '428021', '428022', '428023', '428027', '428029', '427666', '427670', 
'428030', '428031', '427680', '427681', '427685', '427689', '427693', '428032', '428033', '428034', 
'427706', '428035', '428039', '428043', '428044', '428045', '428046', '428047', '427725', '428048', 
'428049', '427736', '427740', '428053', '427744', '428054', '428055', '428056', '427757', '427761', 
'427765', '487513', '428057', '428058', '427775', '428059', '428060', '428061', '428062', '428063', 
]
report_ids2 = [ 
'427794', '427798', '427802', '427806', '427810', '427814', '427818', '428064', '428065', '427826', 
'428067', '427832', '427836', '428068', '427843', '427844', '427848', '427852', '427855', '427860', 
'428069', '496605', '427867', '427871', '428070', '428071', '428072', '428075', '428076', '427892', 
'427896', '427900', '426445', '426449', '426453', '426457', '426461', '426465', '426469', '427105', 
'426476', '427106', '426483', '426487', '427107', '427110', '426495', '427111', '427112', '427113', 
'427114', '427115', '427116', '426515', '427117', '426524', '426528', '427118', '427121', '427122', 
'426536', '426540', '426544', '427123', '426551', '427124', '426558', '427126', '426564', '427127', 
'426571', '427128', '426578', '426582', '427129', '426589', '427130', '427134', '427138', '427142', 
'426596', '427146', '427150', '427151', '427152', '427153', '427154', '427155', '427156', '427157', 
'427158', '427159', '426624', '426628', '427163', '427166', '426633', '426637', '427169', '426641', 
'426644', '427170', '427173', '426653', '427174', '427175', '426663', '427176', '427177', '426673', 
'426677', '426681', '427178', '427179', '427180', '427181', '427182', '427183', '427184', '426706', 
'426710', '426714', '427185', '427186', '426724', '426728', '426732', '427187', '426739', '426743', 
'427188', '426750', '426754', '427192', '427193', '426761', '426768', '427194', '426775', '426779', 
'426783', '427195', '426790', '426794', '427196', '426801', '427198', '426807', '427199', '427201', 
'427202', '427203', '426822', '427206', '426827', '427207', '427208', '427209', '426840', '427210', 
'427211', '427212', '427216', '458515', '427217', '426856', '426860', '427218', '427219', '426870', 
'426871', '426873', '426877', '426881', '427220', '427221', '427222', '426894', '427223', '427225', 
'427227', '427228', '427229', '427230', '427231', '426917', '427232', '426924', '427233', '426931', 
'427234', '426938', '427238', '426943', '426947', '427239', '427240', '426957', '426961', '426965', 
'427241', '427242', '426975', '427243', '427244', '426985', '427245', '427246', '426995', '426999', 
'427003', '427007', '427011', '427015', '427019', '427247', '427026', '427250', '427031', '427035', 
'427251', '427042', '427046', '427050', '427054', '427058', '427252', '427065', '427069', '427076', 
'427253', '427254', '427086', '427255', '427256', '427096', '427100', '427104', '425633', '425637', 
'425641', '425645', '425649', '425653', '425657', '425661', '425665', '425669', '425673', '425677', 
'425681', '425685', '425689', '425693', '425697', '425701', '425705', '425709', '425713', '425717', 
'425718', '425725', '425729', '425733', '425734', '425738', '425742', '425746', '425750', '425754', 
'425758', '425762', '425766', '425770', '425774', '425778', '425782', '425786', '425790', '425794', 
'425798', '425802', '425806', '425810', '425814', '425818', '425822', '425826', '425830', '425834', 
'425838', '425842', '425846', '425850', '425854', '425858', '425862', '425866', '425870', '425874', 
'425878', '425882', '425886', '425890', '425894', '425896', '425900', '425904', '425908', '425912', 
'425916', '425920', '425924', '425928', '425932', '425936', '425940', '425944', '425948', '425952', 
'425956', '425960', '425964', '425968', '425969', '425973', '425977', '425981', '425985', '425989', 
'425993', '425997', '426001', '426005', '426009', '426013', '426017', '426021', '426025', '426029',  
'426033', '426040', '426044', '426048', '426052', '426056', '426060', '426064', '426068', '426072', 
'426076', '426080', '426084', '426088', '426092', '426096', '426100', '426104', '426108', '426112', 
'426116', '426120', '426124', '426128', '426132', '426136', '426140', '426144', '426148', '426152', 
'426156', '426160', '426164', '426168', '426172', '426173', '426177', '426181', '426185', '426189', 
'426193', '426197', '426201', '426205', '426209', '426213', '426217', '426221', '426225', '426229', 
'426233', '426237', '426241', '426245', '426249', '426253', '426257', '426261', '426265', '426269', 
'426273', '426277', '426281', '426285', '426289', '426293', '426297', '426301', '426305', '426309', 
'426313', '426317', '426321', '426325', '426329', '426333', '426337', '426341', '426345', '426349', 
'426353', '426357', '426361', '426365', '426369', '426373', '426377', '426381', '426385', '426389', 
'426393', '426397', '426401', '426405', '426409', '426413', '426417', '426421', '426425', '426429', 
'426433', '426437', '426441', '424830', '424834', '424838', '424842', '424846', '424850', '424854', 
'424858', '424862', '424866', '424870', '424874', '424878', '424882', '424886', '424890', '424894', 
'424898', '424902', '424906', '424910', '424914', '424918', '424922', '424926', '424930', '424934', 
'424938', '424939', '424943', '424947', '424951', '424955', '424959', '424963', '424967', '424971', 
'424975', '424979', '424983', '424987', '424991', '424995', '424999', '425003', '425007', '425011', 
'425015', '425019', '425023', '425027', '425031', '425035', '425039', '425043', '425047', '425051', 
'425055', '425059', '425063', '425067', '425071', '425075', '425079', '425083', '425087', '425091', 
'425095', '425099', '425106', '425110', '425114', '425118', '425122', '425126', '425130', '425134', 
'425137', '425141', '425145', '425149', '425153', '425157', '425161', '425165', '425169', '425173', 
'425177', '425181', '425185', '425189', '425193', '425197', '425201', '425205', '425209', '425213', 
'425217', '425221', '425225', '425229', '425236', '425240', '425244', '425248', '425252', '425256', 
'425260', '425264', '425268', '425269', '425273', '425277', '425281', '425285', '425289', '425293', 
'425297', '425301', '425305', '425309', '425313', '425317', '425321', '425325', '425329', '425333', 
'425337', '425341', '425345', '425349', '425353', '425357', '425361', '425365', '425368', '425372', 
'425376', '425380', '425384', '425388', '425392', '425396', '425400', '425404', '425408', '425412', 
'425416', '425420', '425424', '425428', '425432', '425436', '425440', '425444', '425448', '425449', 
'425453', '425457', '425461', '425465', '425469', '425473', '425477', '425481', '425485', '425489', 
'425493', '425497', '425501', '425505', '425509', '425513', '425517', '425521', '425525', '425529', 
'425533', '425537', '425541', '425545', '425549', '425553', '425557', '425561', '425565', '425569', 
'425573', '425577', '425581', '425585', '425589', '425593', '425597', '425601', '425605', '425609', 
'425613', '425617', '425621', '425625', '425629', '424031', '424035', '424039', '424043', '424047', 
'424051', '424055', '424059', '424063', '424067', '424071', '424075', '424079', '424083', '424087', 
'424091', '424095', '424099', '424103', '424107', '424111', '424115', '424119', '424123', '424127', 
'424131', '424135', '424136', '424140', '424144', '424148', '424152', '424156', '424160', '424164', 
'424168', '424172', '424176', '424180', '424184', '424188', '424192', '424196', '424200', '424204', 
'424208', '424212', '424216', '424220', '424224', '424228', '424232', '424236', '424237', '424241', 
'424245', '424249', '424253', '424257', '424261', '424265', '424269', '424273', '424277', '424281', 
'424285', '424289', '424293', '424297', '424301', '424305', '424309', '424313', '424317', '424321', 
'424325', '424329', '424333', '424337', '424341', '424345', '424349', '424353', '424357', '424361', 
'424365', '424369', '424373', '424377', '424381', '424385', '424389', '424393', '424397', '424401', 
'424405', '424409', '424413', '424417', '424424', '424428', '424432', '424439', '424443', '424447', 
'424451', '424455', '424459', '424463', '424467', '424471', '424475', '424479', '424483', '424487', 
'424491', '424495', '424499', '424503', '424507', '424511', '424515', '424519', '424523', '424527', 
'424531', '424535', '424539', '424543', '424547', '424551', '424555', '424559', '424563', '424567', 
'424571', '424572', '424576', '424580', '424584', '424588', '424592', '424596', '424600', '424604', 
'424608', '424612', '424616', '424620', '424624', '424628', '424632', '424636', '424640', '424644', 
'424648', '424652', '424656', '424660', '424664', '424668', '424672', '424676', '424680', '424684', 
'424688', '424692', '424696', '424700', '424704', '424708', '424712', '424716', '424720', '424724', 
]
report_ids3 = [  
'424728', '424732', '424736', '424740', '424744', '424748', '424752', '424756', '424760', '424764', 
'424765', '424772', '424776', '424780', '424784', '424788', '424792', '424796', '424800', '424804', 
'424808', '424812', '424816', '424820', '424824', '424828', '423222', '423226', '423230', '423234', 
'423238', '423242', '423246', '423250', '423254', '423258', '423262', '423266', '423267', '423271', 
'423275', '423279', '423283', '423287', '423291', '423295', '423299', '423303', '423307', '423311', 
'423315', '423319', '423323', '423327', '423328', '423332', '423336', '423340', '423344', '423348', 
'423352', '423356', '423360', '423364', '423368', '423372', '423376', '423380', '423384', '423388', 
'423392', '423396', '423400', '423404', '423408', '423412', '423416', '423420', '423424', '423428', 
'423433', '423437', '423441', '423445', '423449', '423453', '423457', '423462', '423466', '423470', 
'423474', '423478', '423482', '423486', '423490', '423494', '423498', '423502', '423508', '423512', 
'423516', '423520', '423524', '423528', '423532', '423536', '423540', '423544', '423548', '423552', 
'423556', '423560', '423564', '423568', '423572', '423576', '423580', '423584', '423588', '423592', 
'423596', '423600', '423604', '423608', '423612', '423616', '423620', '423624', '423628', '423635', 
'423639', '423643', '423647', '423651', '423655', '423659', '423663', '423667', '423671', '423675', 
'423679', '423683', '423687', '423691', '423695', '423699', '423703', '423707', '423711', '423715', 
'423719', '423723', '423727', '423731', '423735', '423739', '423743', '423747', '423751', '423755', 
'423759', '423763', '423767', '423768', '423772', '423776', '423780', '423784', '423788', '423792', 
'423796', '423800', '423804', '423808', '423812', '423816', '423820', '423824', '423830', '423834', 
'423838', '423842', '423846', '423850', '423854', '423858', '423862', '423866', '423870', '423874', 
'423878', '423883', '423887', '423891', '423895', '423899', '423903', '423907', '423911', '423915', 
'423919', '423923', '423927', '423931', '423935', '423939', '423943', '423947', '423951', '423955', 
'423959', '423963', '423966', '423971', '423975', '423979', '423983', '423987', '423991', '423995', 
'423999', '424003', '424007', '424011', '424015', '424019', '424023', '424027', '422408', '422412', 
'422416', '422420', '422424', '422428', '422432', '422436', '422440', '422444', '422448', '422449', 
'422453', '422457', '422464', '422468', '422472', '422476', '422480', '422484', '422489', '422493', 
'422497', '422501', '422505', '422509', '422513', '422518', '422519', '422523', '422527', '422531', 
'422534', '422535', '422539', '422543', '422547', '422551', '422555', '422559', '422563', '422567', 
'422571', '422574', '422578', '422582', '422586', '422590', '422593', '422597', '422601', '422605', 
'422608', '422612', '422616', '422623', '422627', '422631', '422635', '422639', '422643', '422647', 
'422651', '422655', '422659', '422664', '422668', '422672', '422676', '422680', '422681', '422688', 
'422692', '422696', '422697', '422701', '422705', '422709', '422713', '422717', '422721', '422725', 
'422729', '422733', '422737', '422741', '422745', '422749', '422753', '422757', '422761', '422765', 
'422769', '422773', '422777', '422781', '422785', '422789', '422793', '422797', '422801', '422803', 
'422809', '422813', '422817', '422821', '422828', '422832', '422836', '422840', '422844', '422848', 
'422852', '422856', '422860', '422864', '422865', '422869', '422873', '422877', '422881', '422885', 
'422889', '422893', '422897', '422901', '422905', '422909', '422913', '422917', '422921', '422925', 
'422929', '422933', '422937', '422941', '422945', '422949', '422953', '422957', '422958', '422962', 
'422966', '422970', '422974', '422978', '422982', '422986', '422990', '422994', '422998', '423002',
'423006', '423010', '423014', '423018', '423022', '423026', '423030', '423034', '423038', '423042', 
'423046', '423050', '423054', '423058', '423062', '423066', '423070', '423074', '423078', '423082', 
'423086', '423090', '423094', '423098', '423102', '423106', '423110', '423114', '423118', '423122', 
'423126', '423130', '423134', '423138', '423142', '423146', '423150', '423154', '423158', '423162', 
'423166', '423170', '423174', '423178', '423182', '423186', '423190', '423194', '423198', '423202', 
'423206', '423210', '423214', '423218', '421208', '421212', '421216', '421220', '421224', '421228', 
'421232', '421236', '421240', '421244', '421248', '421249', '421253', '421257', '421261', '421265', 
'421269', '421273', '421277', '421281', '421285', '421289', '421293', '421297', '421301', '421305', 
'421309', '421313', '421314', '421318', '421322', '421326', '421327', '421331', '421335', '421339', 
'421343', '421347', '421348', '421352', '421356', '421360', '421364', '421368', '421372', '421376', 
'421380', '421384', '421388', '421392', '421393', '421397', '421401', '421405', '421409', '421413', 
'421417', '421421', '421425', '421429', '421433', '421437', '421441', '421445', '421449', '421453', 
'421457', '421461', '421465', '421469', '421473', '421477', '421481', '421485', '421489', '421493', 
'421497', '421501', '421505', '421509', '421513', '421517', '421521', '421525', '421529', '421533', 
'421537', '421541', '421545', '421549', '421553', '421557', '421561', '421565', '421569', '421573', 
'421577', '421581', '421585', '421589', '421593', '421597', '421601', '421605', '421609', '421613', 
'421617', '421621', '421625', '421632', '421636', '421640', '421644', '421648', '421652', '421656', 
'421660', '421664', '421668', '421669', '421673', '421677', '421681', '421685', '421689', '421693', 
'421697', '421701', '421705', '421709', '421713', '421717', '421721', '421725', '421729', '421733', 
'421737', '421741', '421745', '421749', '421753', '421757', '421761', '421762', '421766', '421770', 
'421774', '421778', '421782', '421786', '421790', '421794', '421798', '421802', '421806', '421810', 
'421814', '421818', '421822', '421826', '421830', '421834', '421838', '421842', '421846', '421850', 
'421854', '421858', '421862', '421866', '421870', '421874', '421878', '421882', '421886', '421890', 
'421894', '421901', '421905', '421909', '421913', '421917', '421921', '421925', '421929', '421933', 
'421937', '421941', '421945', '421949', '421953', '421957', '421961', '421965', '421969', '421973', 
'421977', '421981', '421985', '421989', '421993', '422380', '422384', '422388', '422392', '422396', 
'422400', '422404', '418202', '418204', '418206', '418208', '418210', '418212', '418214', '418216', 
'418218', '418220', '418222', '418223', '418225', '418227', '418229', '418231', '418233', '418235', 
'418237', '418239', '418241', '418243', '418245', '418247', '418249', '418251', '418253', '418255', 
'418256', '418258', '418260', '418262', '418263', '418265', '418267', '418269', '418271', '418273', 
'418275', '418277', '418280', '418282', '418284', '418286', '418288', '418290', '418292', '418294', 
'418296', '418298', '418300', '418302', '418304', '418306', '418307', '418309', '418311', '418313', 
'418315', '418317', '418319', '418321', '418323', '418325', '418327', '418329', '418331', '418333', 
'420927', '420929', '420931', '420933', '420935', '420937', '420939', '420941', '420943', '420945', 
'420947', '420949', '420951', '420953', '420955', '420957', '420959', '420961', '420963', '420965', 
'420967', '420969', '420971', '420973', '420975', '420977', '420979', '420981', '420983', '420985', 
'420987', '420989', '420991', '420993', '420995', '420997', '420999', '421001', '421003', '421006', 
'421008', '421010', '421012', '421014', '421016', '421018', '421020', '421022', '421024', '421025', 
'421027', '421029', '421031', '421033', '421035', '421037', '421039', '421041', '421043', '421045', 
'421047', '421049', '421051', '421053', '421055', '421057', '421059', '421061', '421063', '421065', 
'421067', '421069', '421071', '421073', '421074', '421076', '421078', '421080', '421082', '421084', 
'421086', '421088', '421090', '421092', '421094', '421096', '421098', '421100', '421102', '421104', 
'421106', '421108', '421110', '421112', '421114', '421116', '421118', '421120', '421122', '421124', 
'421126', '421128', '421130', '421132', '421134', '421136', '421138', '421140', '421142', '421144', 
'421146', '421148', '421150', '421152', '421154', '421156', '421158', '421160', '421162', '421164', 
'421166', '421168', '421170', '421172', '421174', '421176', '421178', '421180', '421182', '421184', 
'421186', '421188', '421190', '421192', '421194', '421196', '421198', '421200', '421202', '421204'
]
full_list = report_ids 
# + report_ids2 + report_ids3

# Define data fields to return (multi-sort based on order)

fields = [ # this is the list of the characteristics of the data being returned by the query
		'report.id',
		'cube.description.sort(ASC)',
		'cube.tree-sequence.sort(ASC)',
		'fact.value',
		'unit',
		'period.fiscal-year.sort(DESC)',
		'period.fiscal-period',
		'cube.primary-local-name',
		'dimensions.count',
		'dimension-pair.sort(ASC)'
        ]

params = { # this is the list of what's being queried against the search endpoint
         'cube.description': ','.join(XBRL_Elements),
         'report.id': ','.join(full_list),  
         'fact.ultimus': 'TRUE',
         'fields': ','.join(fields)
         }

# Execute the query with loop for all results
endpoint = 'cube'
search_endpoint = 'https://api.xbrl.us/api/v1/' + endpoint + '/search'
orig_fields = params['fields']

count = 0
query_start = datetime.now()
printed = False
while True:
    if not printed:
        printed = True
    res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(access_token)})
    res_json = res.json()
    if 'error' in res_json:
        print('There was an error: {}'.format(res_json['error_description']))
        break

    print("up to", str(offset_value + res_json['paging']['limit']), "records are found so far ...")

    res_df += res_json['data']

    if res_json['paging']['count'] < res_json['paging']['limit']:
        print(" - this set contained fewer than the", res_json['paging']['limit'], "possible, only", str(res_json['paging']['count']), "records.")
        break
    else: 
        offset_value += res_json['paging']['limit'] 
        if 100 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 10 * res_json['paging']['limit']:
                        break 
        elif 500 == res_json['paging']['limit']:
                params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)
                if offset_value == 4 * res_json['paging']['limit']:
                        break 
        params['fields'] = orig_fields + ',' + endpoint + '.offset({})'.format(offset_value)

if not 'error' in res_json:
    current_datetime = datetime.now().replace(microsecond=0)
    time_taken = current_datetime - query_start
    index = pd.DataFrame(res_df).index
    total_rows = len(index)
    your_limit = res_json['paging']['limit']
    limit_message = "If the results below match the limit noted above, you might not be seeing all rows, and should consider upgrading (https://xbrl.us/access-token).\n"
    
    if your_limit == 100:
        print("\nThis non-Member account has a limit of " , 10 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    elif your_limit == 500:
        print("\nThis Basic Individual Member account has a limit of ", 4 * your_limit, " rows per query from our Public Filings Database. " + limit_message)
    
    print("\nAt " + current_datetime.strftime("%c") +  ", the query finished with  ", str(total_rows), "  rows returned in " + str(time_taken) + " for \n" +  urllib.parse.unquote(res.url))
    
    
    df = pd.DataFrame(res_df)
    # the format truncates the HTML display of numerical values to two decimals; .csv data is unaffected
    pd.options.display.float_format = '{:,.2f}'.format
    display(HTML(df.to_html()))

up to 5000 records are found so far ...
up to 10000 records are found so far ...
up to 15000 records are found so far ...
up to 20000 records are found so far ...
up to 25000 records are found so far ...
 - this set contained fewer than the 5000 possible, only 3812 records.

At Tue May  2 12:57:04 2023, the query finished with   23812   rows returned in 0:00:41.673013 for 
https://api.xbrl.us/api/v1/cube/search?cube.description=200+-+Schedule+-+Summary+of+Utility+Plant+and+Accumulated+Provisions+for+Depreciation,+Amortization+and+Depletion&report.id=568937,570499,570510,570501,570507,570504,568947,563267,563272,573288,564870,571058,570426,564792,563256,563278,563213,569115,568436,570532,563364,564700,569108,563355,563352,563358,563346,568435,563114,568431,561458,563112,569845,569738,564872,570272,570679,563307,563361,568432,564698,563340,569686,570242,568219,563304,568551,570448,570263,563124,569286,570544,570405,570408,569871,563216,569301,570388,570490,563230,563207,570484,569827,567

,report.id,cube.description,cube.tree-sequence,fact.value,unit,period.fiscal-year,period.fiscal-period,cube.primary-local-name,dimensions.count,dimension-pair
0,570544,"200 - Schedule - Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion",3,6069793,USD,2022,Y,UtilityPlantInServiceClassified,1,[{'UtilityTypeAxis': 'ElectricUtilityMember'}]
1,569301,"200 - Schedule - Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion",3,415678164,USD,2022,Y,UtilityPlantInServiceClassified,1,[{'UtilityTypeAxis': 'ElectricUtilityMember'}]
2,569326,"200 - Schedule - Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion",3,4795295870,USD,2022,Y,UtilityPlantInServiceClassified,1,[{'UtilityTypeAxis': 'GasUtilityMember'}]
23809,427449,"200 - Schedule - Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion",34,288367311,USD,2018,Y,AccumulatedProvisionForDepreciationAmortizationAndDepletionOfPlantUtility,0,
23810,427581,"200 - Schedule - Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion",34,1300350722,USD,2018,Y,AccumulatedProvisionForDepreciationAmortizationAndDepletionOfPlantUtility,0,
23811,427919,"200 - Schedule - Summary of Utility Plant and Accumulated Provisions for Depreciation, Amortization and Depletion",34,491176799,USD,2018,Y,AccumulatedProvisionForDepreciationAmortizationAndDepletionOfPlantUtility,0,


In [ ]:
# If you run this program locally, you can save the output to a file on your computer (modify D:\results.csv to your system)
df.to_csv(r"D:\results.csv",sep=",")